# Pose Classification Training in Google Colab

This notebook trains a lightweight pose classifier to distinguish **bersedia** (ready) vs **berlari** (running) using MediaPipe for pose extraction and TensorFlow Keras for a tiny dense model. The resulting TFLite model can be integrated into the Flutter app.

## 1️⃣ Setup – Install dependencies

In [ ]:
!pip install -q tensorflow mediapipe opencv-python scikit-learn tqdm

## 2️⃣ Download MediaPipe Pose Landmarker model

The new `mediapipe.tasks` API requires a `.task` model file (replaces the old built-in BlazePose).

In [ ]:
# Download the pose landmarker model bundle from Google
!wget -q -O pose_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task
print('✅ Model downloaded: pose_landmarker.task')

## 3️⃣ Prepare training data

Create a folder `data/` with two subfolders: `bersedia/` and `berlari/`.
Upload at least 20‑30 images per class (full‑body poses, clear lighting).
The folder structure must look like:
```
data/
├─ bersedia/
│  ├─ img1.jpg
│  └─ ...
└─ berlari/
   ├─ img1.jpg
   └─ ...
```

In [ ]:
from google.colab import files

print('Upload a zip of your `data/` folder (e.g., data.zip)')
uploaded = files.upload()

import zipfile, os
for fn in uploaded.keys():
    with zipfile.ZipFile(fn, 'r') as zip_ref:
        zip_ref.extractall('.')
    print(f'Extracted {fn}')

## 4️⃣ Training script (uses new `mediapipe.tasks` API)

In [ ]:
import os, json
import numpy as np
import cv2
import mediapipe as mp
import tensorflow as tf
from sklearn.model_selection import train_test_split

print(f'mediapipe version : {mp.__version__}')
print(f'tensorflow version: {tf.__version__}')

# ── New mediapipe.tasks API ───────────────────────────────────────────────────
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

BaseOptions       = mp.tasks.BaseOptions
PoseLandmarker    = mp_vision.PoseLandmarker
PoseLandmarkerOptions = mp_vision.PoseLandmarkerOptions
VisionRunningMode = mp_vision.RunningMode

MODEL_PATH = 'pose_landmarker.task'

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

DATA_DIR = 'dataset'
BONE_CONNECTIONS = [
    (11, 13), (13, 15), (12, 14), (14, 16), (11, 12), (11, 23), (12, 24),
    (23, 25), (25, 27), (24, 26), (26, 28), (23, 24)
]

def extract_features(img_path, landmarker):
    """Extract normalized pose features from an image using the new tasks API."""
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None

    # Convert BGR → RGB numpy array, then wrap in mp.Image
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

    result = landmarker.detect(mp_image)

    # result.pose_landmarks is a list of lists (one per detected person)
    if not result.pose_landmarks:
        return None

    lm = result.pose_landmarks[0]  # first (and only) person
    coords = np.array([(p.x, p.y, p.z) for p in lm], dtype=np.float32)

    # Normalize: center on hip midpoint, scale by torso length
    hip_center = (coords[23] + coords[24]) / 2
    coords -= hip_center
    torso_len = np.linalg.norm((coords[11] + coords[12]) / 2)
    if torso_len > 1e-6:
        coords /= torso_len

    features = list(coords.flatten())
    for s, e in BONE_CONNECTIONS:
        features.extend(coords[e] - coords[s])
    for s, e in BONE_CONNECTIONS:
        vec = coords[e] - coords[s]
        features.append(float(np.dot(vec, vec)))

    return np.array(features, dtype=np.float32)

# ── Build dataset ─────────────────────────────────────────────────────────────
X, y = [], []
label_map = {}
skipped = 0

with PoseLandmarker.create_from_options(options) as landmarker:
    for idx, label in enumerate(sorted(os.listdir(DATA_DIR))):
        label_path = os.path.join(DATA_DIR, label)
        if not os.path.isdir(label_path):
            continue
        label_map[label] = idx
        for f in os.listdir(label_path):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                fp = os.path.join(label_path, f)
                feats = extract_features(fp, landmarker)
                if feats is not None:
                    X.append(feats)
                    y.append(idx)
                else:
                    skipped += 1

print(f'Dataset : {len(X)} samples, {skipped} skipped (no pose detected)')
print(f'Labels  : {label_map}')

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Model ─────────────────────────────────────────────────────────────────────
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(label_map), activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# ── Export TFLite ─────────────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('pose_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

with open('pose_labels.json', 'w') as f:
    json.dump(label_map, f)

print('✅ Training complete.')
print(f'   Feature vector size : {X.shape[1]}')
print('   Output files        : pose_classifier.tflite, pose_labels.json')

## 5️⃣ Download the model files

In [ ]:
from google.colab import files
files.download('pose_classifier.tflite')
files.download('pose_labels.json')

## 6️⃣ Integrate into Flutter

1. Copy `pose_classifier.tflite` and `pose_labels.json` into `assets/models/` in your Flutter project.
2. Add them to `pubspec.yaml` under `assets:`.
3. Run `flutter pub get`.
4. Use the provided `lib/ml/pose_classifier.dart` to run inference.